# AIML-04 · ONNX vs TensorRT benchmark (NVIDIA GPU required)

**Run this on Google Colab or Kaggle with a GPU runtime** (Colab: *Runtime → Change runtime type → T4 GPU*; Kaggle: *Settings → Accelerator → GPU*).
Your laptop has no NVIDIA GPU, so TensorRT can **not** run there.

What this notebook does, in order:
1. confirms an NVIDIA GPU is present (stops otherwise)
2. benchmarks the **PyTorch** model on GPU (baseline)
3. exports to **ONNX** and benchmarks it with ONNX Runtime
4. exports to a **TensorRT FP16 engine** and benchmarks it
5. checks the three backends give similar detections (sanity)
6. saves `benchmark_gpu.json` + a table — **all numbers are measured at run time on whatever GPU you get; nothing is pre-filled**

> Status in the project docs: this notebook is *written but not executed by the author of the ZIP*. After you run it, paste your own table into `docs/BENCHMARK_REPORT.md`.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU found. Switch the runtime to GPU and run again."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# ultralytics installs/uses TensorRT itself during export. If pip upgrades packages, restart the runtime once and re-run from here.
!pip -q install ultralytics onnx onnxslim onnxruntime-gpu pandas

In [ ]:
import time, json, platform, numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = "yolo11n.pt"      # or upload your fine-tuned best.pt and put its filename here
IMGSZ   = 640
WARMUP, RUNS = 20, 200

import urllib.request
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "bus.jpg")
IMG = "bus.jpg"

In [ ]:
def bench(fn, warmup=WARMUP, runs=RUNS):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    ts = []
    for _ in range(runs):
        t = time.perf_counter(); fn(); torch.cuda.synchronize(); ts.append((time.perf_counter() - t) * 1000)
    a = np.array(ts)
    return {"mean_ms": float(a.mean()), "median_ms": float(np.median(a)), "p95_ms": float(np.percentile(a, 95)),
            "fps_mean": float(1000 / a.mean()), "runs": runs}

results, detections = {}, {}
m = YOLO(WEIGHTS)
results["pytorch_fp32_gpu"] = bench(lambda: m.predict(IMG, imgsz=IMGSZ, device=0, half=False, verbose=False))
detections["pytorch_fp32_gpu"] = len(m.predict(IMG, imgsz=IMGSZ, device=0, conf=0.25, verbose=False)[0].boxes)
results["pytorch_fp32_gpu"]

In [ ]:
# ONNX (this is the ONNX test, NOT TensorRT)
onnx_path = m.export(format="onnx", imgsz=IMGSZ, simplify=True, opset=17)
mo = YOLO(onnx_path)
results["onnx_gpu_or_cpu"] = bench(lambda: mo.predict(IMG, imgsz=IMGSZ, device=0, verbose=False))
detections["onnx_gpu_or_cpu"] = len(mo.predict(IMG, imgsz=IMGSZ, device=0, conf=0.25, verbose=False)[0].boxes)
import onnxruntime as ort
print("ONNX Runtime providers available:", ort.get_available_providers())
results["onnx_gpu_or_cpu"]

In [ ]:
# TensorRT FP16 engine (this is the TensorRT test). Export can take a few minutes.
engine_path = m.export(format="engine", imgsz=IMGSZ, half=True, device=0, workspace=4)
me = YOLO(engine_path)
results["tensorrt_fp16"] = bench(lambda: me.predict(IMG, imgsz=IMGSZ, device=0, verbose=False))
detections["tensorrt_fp16"] = len(me.predict(IMG, imgsz=IMGSZ, device=0, conf=0.25, verbose=False)[0].boxes)
results["tensorrt_fp16"]

In [ ]:
df = pd.DataFrame(results).T
base = df.loc["pytorch_fp32_gpu", "mean_ms"]
df["speedup_vs_pytorch"] = base / df["mean_ms"]
df["n_detections_conf0.25"] = pd.Series(detections)
display(df.round(2))

In [ ]:
import tensorrt
info = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "tensorrt": tensorrt.__version__,
        "onnxruntime": ort.__version__, "python": platform.python_version(), "weights": WEIGHTS, "imgsz": IMGSZ,
        "warmup": WARMUP, "runs": RUNS, "image": IMG,
        "note": "End-to-end predict() latency incl. pre/post-processing, batch 1. Valid only for this GPU/software combination."}
out = {"info": info, "results": results, "detections": detections}
Path("benchmark_gpu.json").write_text(json.dumps(out, indent=2))
ax = df["mean_ms"].plot.bar(title=f"Mean latency (ms) - {info['gpu']}", rot=20); ax.set_ylabel("ms")
from google.colab import files  # on Kaggle: the files are in the output tab instead
files.download("benchmark_gpu.json")

### How to read the result
* `speedup_vs_pytorch` is computed from **your** measurements. Do not quote numbers from anyone else's run.
* The detection counts should be similar across backends (small differences are normal, e.g. FP16). A big mismatch means a broken export - investigate before trusting the timings.
* Copy `benchmark_gpu.json` into `outputs/metrics/` in the project and update `docs/BENCHMARK_REPORT.md`.